In [9]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
import joblib
from gene_thesaurus import GeneThesaurus

In [10]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-gex")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.gex.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

output_dir = "/home/mariya/Data/pallas/models/gex"
random_seed = 42
n_jobs = 10
selected_features_path = os.path.join(output_dir, "selected_features.txt")
pipeline_path = os.path.join(output_dir, "pipeline.pkl")
hyperparams_path = os.path.join(output_dir, "hyperparams.pkl")

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

# Hardcoded variables
FEATURE_SELECTION_PERCENTILE=30  # KEEP TOP n% OF FEATURES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-gex/features.gex.csv
y_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-gex/labels.csv
output_dir: /home/mariya/Data/pallas/models/gex
random_seed: 42
n_jobs: 10
selected_features_path: /home/mariya/Data/pallas/models/gex/selected_features.txt
pipeline_path: /home/mariya/Data/pallas/models/gex/pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/models/gex/hyperparams.pkl


In [11]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000288825,ENSG00000288859,ENSG00000288867,ENSG00000289549,ENSG00000289604,ENSG00000290292,ENSG00000291237,ENSG00000293137,ENSG00000293560,ENSG00000310560
id,,,,,,,,,,,,,,,,,,,,,
16-001,-0.878920,-1.117704,3.805587,3.811056,4.273350,4.424479,0.425722,1.391791,4.590267,5.211272,...,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-0.000571,-1.117704,5.225302
16-002,-0.587219,-1.117704,4.203626,2.386586,3.924283,3.726373,0.479800,2.579887,4.143098,5.429659,...,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-0.825001,-0.804777,5.646595
16-006,-0.836400,-1.117704,3.978518,3.891260,3.466091,3.943861,-0.572943,2.262237,3.788627,5.558767,...,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,0.474879,-1.117704,6.157250
16-008,3.900577,-1.117704,3.865507,4.327653,3.190645,5.199584,0.171746,2.730728,5.390347,5.834229,...,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,2.291071,-1.117704,7.849882
16-015,-0.116930,-1.117704,4.884844,3.153458,2.317838,5.047187,0.147842,2.460700,3.175844,4.154182,...,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-1.117704,-0.495035,-1.117704,5.402267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-AB-2991,-1.117704,-0.610606,3.997347,3.183419,3.615880,0.566971,2.124191,3.765116,4.218102,6.677854,...,1.384492,0.514680,-1.117704,-1.117704,-1.117704,4.219335,4.411309,-0.090163,0.576852,5.728160
TCGA-AB-2998,-1.117704,-1.117704,4.124676,3.693561,4.061581,-0.330406,5.223262,4.116433,4.768916,5.500750,...,2.244568,3.441028,-0.394300,-0.394300,-0.157111,4.903072,4.376972,-0.248527,1.312982,5.191969
TCGA-AB-2999,-1.117704,-1.117704,4.733386,3.643477,2.655369,0.774984,1.370865,3.742882,4.378186,6.035298,...,1.559973,1.620045,1.752127,-1.117704,-1.117704,4.503311,4.606782,-1.117704,0.654414,5.882896


In [12]:
y_train = pd.read_csv(y_train_path, index_col=0)
y_train

,class
id,
16-001,"B-ALL, ETV6::RUNX1"
16-002,"B-ALL, ETV6::RUNX1"
16-006,"B-ALL, ETV6::RUNX1"
16-008,T-ALL
16-015,"B-ALL, high hyperdiploid"
...,...
TCGA-AB-2991,"AML, APL-PML::RARA"
TCGA-AB-2998,"AML, APL-PML::RARA"
TCGA-AB-2999,"AML, APL-PML::RARA"


In [13]:
print("y_train class distribution:")
print(y_train.value_counts())

y_train class distribution:
class                   
B-ALL, high hyperdiploid    389
AML, KMT2A-r                365
B-ALL, ETV6::RUNX1          323
B-ALL, BCR::ABL1-like       261
AML, RUNX1::RUNX1T1         253
AML, NPM1alt                159
B-ALL, KMT2A-r              152
B-ALL, PAX5alt              141
T-ALL                       125
AML, CBFB::MYH11            117
B-ALL, DUX4-r               115
B-ALL, BCR::ABL1             97
B-ALL, TCF3::PBX1            88
B-ALL, hypodiploid           87
AML, CEBPAalt                79
AML, NUP98-r                 67
B-ALL, ZNF384-r              56
B-ALL, ETV6::RUNX1-like      43
B-ALL, MEF2D-r               40
B-ALL, iAMP21                36
AML, DEK::NUP214             31
B-ALL, PAX5 P80R             27
AML, CBFA2T3::GLIS2          25
Non-leukemic                 15
AML, APL-PML::RARA           14
B-ALL, NUTM1-r               13
B-ALL, MYC-r                 12
AML, KAT6A-r                  8
AML, ETV6-r                   8
B-ALL, HLF-r       

In [14]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(5)

In [15]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array

In [16]:
# Reduce memory usage by converting to float32
X_train = X_train.astype(np.float32)

In [ ]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS
N_FOLDS_INITIAL=3 # down from min_class_count to speed up initial search

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2', 0.25],
    'rf__min_samples_split': [5, 10, 20],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': ['balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=N_FOLDS_INITIAL,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=1,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_


Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best hyperparameters: {'rf__n_estimators': 50, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 1, 'rf__max_features': 0.25, 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9196732820981938


FileNotFoundError: [Errno 2] No such file or directory: '/home/mariya/Data/pallas/models/gex/hyperparams.pkl'

In [18]:

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparams_path)

# Save features
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = X_train.columns[selected_mask].tolist()

# Write selected features to a text file
with open(selected_features_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

['/home/mariya/Data/pallas/models/gex/pipeline.pkl']

In [19]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparams_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=1,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, os.path.join(output_dir, "final_model.pkl"))

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.25, 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed: 422.7min
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed: 431.1min finished


Cross-validation F1-weighted: 0.924 ± 0.008


['/home/mariya/Data/pallas/models/gex/final_model.pkl']